# 数据下载

本Notebook完成所有数据的下载工作，包括：
1. 10只股票的后复权日度行情数据
2. 沪深300指数（000300）日度数据
3. 创业板指（399006）日度数据（自选指数）
4. CPI同比增速（宏观指标）
5. M2同比增速（自选宏观指标）
6. 10只股票的财务指标

所有下载结果记录到 `download_log.txt`。

## 1. 环境准备

In [ ]:
import os
import akshare as ak
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import warnings

warnings.filterwarnings('ignore')

# 设置pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# 打印akshare版本
print(f"akshare版本: {ak.__version__}")
print(f"pandas版本: {pd.__version__}")

## 2. 定义股票列表和参数

In [ ]:
# 10只股票列表（代码: 名称 映射）
stock_list = {
    '603063': '禾望电气',
    '002518': '科士达',
    '603118': '共进股份',
    '000063': '中兴通讯',
    '600036': '招商银行',
    '002142': '宁波银行',
    '002594': '比亚迪',
    '000002': '万科A',
    '600519': '贵州茅台',
    '002352': '顺丰控股'
}

# 股票行业映射
stock_industry = {
    '603063': '能源',
    '002518': '能源',
    '603118': '通讯',
    '000063': '通讯',
    '600036': '银行',
    '002142': '银行',
    '002594': '汽车',
    '000002': '房地产',
    '600519': '白酒',
    '002352': '物流'
}

# 时间范围
start_date = '20200101'
end_date = datetime.now().strftime('%Y%m%d')

print(f"数据时间范围: {start_date} 至 {end_date}")
print(f"股票数量: {len(stock_list)}")
print(f"行业分布: {dict(pd.Series(stock_industry).value_counts())}")

## 3. 定义下载日志函数

In [ ]:
def log_download(log_file, status, data_type, code_or_name, shape=None, error_msg=None):
    """
    记录下载日志
    
    参数:
        log_file: 日志文件路径
        status: 'SUCCESS' 或 'FAILED'
        data_type: 数据类型（如 'stock', 'index'）
        code_or_name: 股票代码或名称
        shape: 数据形状（成功时）
        error_msg: 错误信息（失败时）
    """
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    if status == 'SUCCESS':
        msg = f"[{timestamp}] SUCCESS  {data_type}_{code_or_name}  shape={shape}\n"
    else:
        msg = f"[{timestamp}] FAILED   {data_type}_{code_or_name}  Error: {error_msg}\n"
    
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write(msg)
    
    print(msg.strip())

## 4. 下载股票行情数据

使用 `ak.stock_zh_a_hist()` 获取后复权日度行情数据。

In [ ]:
# 定义保存路径
stock_dir = 'data/stock'
log_file = 'download_log.txt'

# 清空日志文件（如果存在）
if os.path.exists(log_file):
    os.remove(log_file)

print("=" * 60)
print("开始下载股票行情数据...")
print("=" * 60)

for code, name in stock_list.items():
    try:
        # 下载后复权数据
        # akshare的adjust参数: '' 不复权, 'qfq' 前复权, 'hfq' 后复权
        df = ak.stock_zh_a_hist(symbol=code, period='daily', 
                                 start_date=start_date, end_date=end_date,
                                 adjust='hfq')
        
        # 重命名列
        df.columns = ['date', 'open', 'close', 'high', 'low', 'volume', 'amount', 
                      'amplitude', 'pct_change', 'change', 'turnover']
        
        # 保留必含字段
        df = df[['date', 'open', 'close', 'high', 'low', 'volume', 'amount']]
        
        # 保存为CSV
        file_path = os.path.join(stock_dir, f'stock_{code}.csv')
        df.to_csv(file_path, index=False, encoding='utf-8-sig')
        
        # 记录日志
        log_download(log_file, 'SUCCESS', 'stock', code, shape=df.shape)
        
        # 暂停避免请求过快
        time.sleep(0.5)
        
    except Exception as e:
        log_download(log_file, 'FAILED', 'stock', code, error_msg=str(e))
        time.sleep(1)

print("\n股票行情数据下载完成！")

## 5. 下载指数数据

- 必选：沪深300（000300）
- 自选：创业板指（399006），反映成长股市场表现

In [ ]:
index_dir = 'data/index'

# 指数列表
index_list = {
    '000300': '沪深300',  # 必选，CAPM市场基准
    '399006': '创业板指'   # 自选，反映成长股市场
}

print("=" * 60)
print("开始下载指数数据...")
print("=" * 60)

for code, name in index_list.items():
    try:
        # 下载指数数据
        df = ak.stock_zh_index_daily(symbol=f'sh{code}') if code.startswith('000') else ak.stock_zh_index_daily(symbol=f'sz{code}')
        
        # 重命名列
        df.columns = ['date', 'open', 'close', 'high', 'low', 'volume', 'amount']
        
        # 筛选时间范围
        df['date'] = pd.to_datetime(df['date'])
        df = df[(df['date'] >= start_date) & (df['date'] <= end_date)]
        df['date'] = df['date'].dt.strftime('%Y-%m-%d')
        
        # 保存
        file_path = os.path.join(index_dir, f'index_{code}.csv')
        df.to_csv(file_path, index=False, encoding='utf-8-sig')
        
        log_download(log_file, 'SUCCESS', 'index', code, shape=df.shape)
        time.sleep(0.5)
        
    except Exception as e:
        log_download(log_file, 'FAILED', 'index', code, error_msg=str(e))
        time.sleep(1)

print("\n指数数据下载完成！")

## 6. 下载宏观经济指标

- 必选：CPI同比增速
- 自选：M2同比增速（反映货币政策宽松程度，与股市流动性相关）

In [ ]:
macro_dir = 'data/macro'

print("=" * 60)
print("开始下载宏观经济指标...")
print("=" * 60)

# 6.1 CPI同比增速
try:
    df_cpi = ak.macro_china_cpi_yearly()
    df_cpi.columns = ['date', 'cpi_yoy']
    df_cpi['date'] = pd.to_datetime(df_cpi['date'])
    df_cpi = df_cpi[(df_cpi['date'] >= '2020-01-01') & (df_cpi['date'] <= end_date)]
    df_cpi['date'] = df_cpi['date'].dt.strftime('%Y-%m-%d')
    
    file_path = os.path.join(macro_dir, 'macro_cpi.csv')
    df_cpi.to_csv(file_path, index=False, encoding='utf-8-sig')
    
    log_download(log_file, 'SUCCESS', 'macro', 'cpi', shape=df_cpi.shape)
    
except Exception as e:
    log_download(log_file, 'FAILED', 'macro', 'cpi', error_msg=str(e))

time.sleep(0.5)

# 6.2 M2同比增速
try:
    df_m2 = ak.macro_china_m2_yearly()
    df_m2.columns = ['date', 'm2_yoy']
    df_m2['date'] = pd.to_datetime(df_m2['date'])
    df_m2 = df_m2[(df_m2['date'] >= '2020-01-01') & (df_m2['date'] <= end_date)]
    df_m2['date'] = df_m2['date'].dt.strftime('%Y-%m-%d')
    
    file_path = os.path.join(macro_dir, 'macro_m2.csv')
    df_m2.to_csv(file_path, index=False, encoding='utf-8-sig')
    
    log_download(log_file, 'SUCCESS', 'macro', 'm2', shape=df_m2.shape)
    
except Exception as e:
    log_download(log_file, 'FAILED', 'macro', 'm2', error_msg=str(e))

print("\n宏观经济指标下载完成！")

## 7. 下载财务指标

获取10只股票最近5个年度的财务指标，包括：
- 净资产收益率（ROE）
- 净利润率

格式要求：长格式，字段为 `code, year, indicator, value`

In [ ]:
finance_dir = 'data/finance'

print("=" * 60)
print("开始下载财务指标...")
print("=" * 60)

# 存储所有财务数据
finance_data = []

for code, name in stock_list.items():
    try:
        # 获取财务分析指标
        df_fin = ak.stock_financial_analysis_indicator(symbol=code)
        
        # 筛选需要的指标
        # 常见指标：净资产收益率、销售净利率、资产负债率等
        # akshare返回的列名可能包含：日期, 净资产收益率, 销售净利率, 等
        
        # 查找包含ROE和净利润率的列
        roe_col = [c for c in df_fin.columns if '净资产收益率' in c or 'ROE' in c.upper()]
        npm_col = [c for c in df_fin.columns if '净利率' in c or '销售净利率' in c]
        
        if len(roe_col) > 0 and len(npm_col) > 0:
            # 提取年份
            df_fin['year'] = pd.to_datetime(df_fin.iloc[:, 0]).dt.year
            
            # 筛选最近5年
            recent_years = list(range(2020, 2025))
            df_fin = df_fin[df_fin['year'].isin(recent_years)]
            
            for _, row in df_fin.iterrows():
                year = row['year']
                # ROE
                roe_val = row[roe_col[0]]
                if pd.notna(roe_val):
                    try:
                        finance_data.append({
                            'code': code,
                            'year': int(year),
                            'indicator': 'ROE',
                            'value': float(roe_val)
                        })
                    except:
                        pass
                
                # 净利润率
                npm_val = row[npm_col[0]]
                if pd.notna(npm_val):
                    try:
                        finance_data.append({
                            'code': code,
                            'year': int(year),
                            'indicator': '净利润率',
                            'value': float(npm_val)
                        })
                    except:
                        pass
        
        log_download(log_file, 'SUCCESS', 'finance', code, shape=(len([d for d in finance_data if d['code']==code]), 4))
        time.sleep(0.5)
        
    except Exception as e:
        log_download(log_file, 'FAILED', 'finance', code, error_msg=str(e))
        time.sleep(1)

# 保存为长格式CSV
df_finance = pd.DataFrame(finance_data)
file_path = os.path.join(finance_dir, 'finance_indicators.csv')
df_finance.to_csv(file_path, index=False, encoding='utf-8-sig')

print(f"\n财务指标下载完成！共 {len(df_finance)} 条记录")
print(df_finance.head(10))

## 8. 检查下载结果

In [ ]:
print("=" * 60)
print("数据下载情况汇总")
print("=" * 60)

# 检查各目录文件数量
dirs = ['data/stock', 'data/index', 'data/macro', 'data/finance']
for d in dirs:
    files = [f for f in os.listdir(d) if f.endswith('.csv')]
    print(f"{d}: {len(files)} 个文件")

# 显示下载日志
print("\n" + "=" * 60)
print("下载日志内容")
print("=" * 60)
with open(log_file, 'r', encoding='utf-8') as f:
    print(f.read())

## 9. 数据预览

快速查看下载的数据格式和内容。

In [ ]:
# 股票数据预览
sample_stock = pd.read_csv('data/stock/stock_600519.csv')
print("贵州茅台数据样例：")
print(sample_stock.head())
print(f"\n数据形状: {sample_stock.shape}")
print(f"数据类型:\n{sample_stock.dtypes}")

In [ ]:
# 指数数据预览
sample_index = pd.read_csv('data/index/index_000300.csv')
print("沪深300数据样例：")
print(sample_index.head())
print(f"\n数据形状: {sample_index.shape}")

In [ ]:
# 宏观数据预览
sample_macro = pd.read_csv('data/macro/macro_cpi.csv')
print("CPI数据样例：")
print(sample_macro.head())
print(f"\n数据形状: {sample_macro.shape}")

In [ ]:
# 财务数据预览
sample_finance = pd.read_csv('data/finance/finance_indicators.csv')
print("财务指标数据样例：")
print(sample_finance.head(20))
print(f"\n数据形状: {sample_finance.shape}")

**数据下载完成！**

下一步：运行 `02_clean.ipynb` 进行数据清洗。